In [1]:
import os
import boto3
from sagemaker import get_execution_role
import time
from pprint import pprint
import shutil

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# function name
str_function_name = 'genxii-pd-starting-feats-2'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

boto3==1.24.59
pandas==1.2.4
scikit_learn==0.24.1
tqdm==4.64.1

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pandas as pd
import numpy as np
import json
import boto3
import time
from tqdm import tqdm

# lambda handler
def lambda_handler(event, context):
    # constants
    str_project = '20231010-gen-xii'
    
    # get df_hyperparameters
    str_filename = 'df_hyperparameters.csv'
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/02_model/12_step_function/{str_filename}'
    df = pd.read_csv(str_uri)
    # convert to dict
    dict_hyperparameters = dict(zip(df['keys'], df['values']))
    
    # get eval metric
    str_eval_metric = dict_hyperparameters['STR_EVAL_METRIC']
    print(f'Eval metric: {str_eval_metric}')
        
    # load output from shared feature selection
    print('Loading output from shared feature selection...')
    str_filename = 'df_iterative_feat_select.csv'
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/01_feat_select/04_batch_feature_selection/results/{str_filename}'
    df_feat_select = pd.read_csv(str_uri)
    
    # get max n feats and remove it
    print('Removing max n feats...')
    int_max_feats = np.max(df_feat_select['n_feats'])
    df_feat_select = df_feat_select[df_feat_select['n_feats'] < int_max_feats]
    
    # logic for sorting
    print('Sorting...')
    if str_eval_metric in ['AUC','PRAUC','F1']:
        bool_ascending = False
    else:
        bool_ascending = True
    df_feat_select.sort_values(by='flt_eval_metric_valid', ascending=bool_ascending, inplace=True)

    # get list of best features
    list_cols_model = eval(df_feat_select['list_cols_model'].iloc[0])
    
    # load in the list of features to drop
    print('Loading list of features to drop...')
    str_filename = 'df_feats_to_drop.csv'
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/02_model/01_lambda_get_starting_feats/{str_filename}'
    try:
        list_feats_drop = list(pd.read_csv(str_uri)['feature'])
    except:
        # create df_feats_to_drop
        list_feats_drop = []
        df_feats_to_drop = pd.DataFrame({'feature': list_feats_drop})
        # write
        df_feats_to_drop.to_csv(str_uri, index=False)
    
    # remove the list of features to drop
    print('Removing the list of features to drop...')
    list_cols_model = [col for col in list_cols_model if col not in list_feats_drop]
    
    # write to s3
    print('Writing list of columns in model to s3...')
    df_cols_in_model = pd.DataFrame({'feature': list_cols_model})
    str_filename = 'df_cols_in_model.csv'
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/02_model/01_lambda_get_starting_feats/{str_filename}'
    df_cols_in_model.to_csv(str_uri, index=False)
    
    # write to s3 as json for map in step function
    print('Writing json of columns in model to s3...')
    str_list_cols_model = json.dumps(list_cols_model)
    cls_client_s3 = boto3.client('s3')
    str_filename = 'json_cols_in_model.json'
    str_key = f'02_pricing_pd/02_model/02_model/01_lambda_get_starting_feats/{str_filename}'
    cls_client_s3.put_object(
        Bucket=str_project,
        Key=str_key,
        Body=str_list_cols_model,
    )

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-pd-starting-feats-2

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  18.94kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
3.8: Pulling from lambda/python
e2d50c2bf7c9: Pulling fs layer
dbfbe01b6453: Pulling fs layer
ff6a272b6451: Pulling fs layer
79a77e7c1be9: Pulling fs layer
52d42e85a88e: Pulling fs layer
df5d7fdf33b4: Pulling fs layer
79a77e7c1be9: Waiting
52d42e85a88e: Waiting
df5d7fdf33b4: Waiting
ff6a272b6451: Download complete
dbfbe01b6453: Verifying Checksum
dbfbe01b6453: Download complete
79a77e7c1be9: Verifying Checksum
79a77e7c1be9: Download complete
df5d7fdf33b4: Verifying Checksum
df5d7fdf33b4: Download complete
52d42e85a88e: Verifying Checksum
52d42e85a88e: Download complete
e2d50c2bf7c9: Verifying Checksum
e2d50c2bf7c9: Download complete
e2d50c2bf7c9: Pull complete
dbfbe01b6453: Pull complete
ff6a272b6451: Pull complete
79a77e7c1be9: Pull complete
52d42e85a88e: Pull complete
df5d7fdf33b4: Pull complete
Digest: sha256:c89f53f63d556ed99c19245c16a0b7af192dd4f65aa1ee07f8684852969c1097
Status: Downlo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 630.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.5/502.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.9/137.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.1/220.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 7.5 MB/s eta 0:00:00
Removing intermediate container 88661be028bc
 ---> 1c45403217ee
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> ea484528b0e4
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Running in 8a6f949d6aa9
Removing intermediate container 8a6f949d6aa9
 ---> 95da2b15b7f6
Succ

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded
{
    "repository": {
        "repositoryArn": "arn:aws:ecr:us-west-2:836690756591:repository/genxii-pd-starting-feats-2",
        "registryId": "836690756591",
        "repositoryName": "genxii-pd-starting-feats-2",
        "repositoryUri": "836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pd-starting-feats-2",
        "createdAt": 1697669472.0,
        "imageTagMutability": "MUTABLE",
        "imageScanningConfiguration": {
            "scanOnPush": true
        },
        "encryptionConfiguration": {
            "encryptionType": "AES256"
        }
    }
}
The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pd-starting-feats-2]
000fcb07a628: Preparing
6801076a9e6d: Preparing
621fd133cada: Preparing
00e37b3ec5b7: Preparing
2ae7c19e0b5c: Preparing
9e6784b558c3: Preparing
15dd6c63f3a2: Preparing
8819b61d0672: Preparing
30349c0bf45d: Preparing
91232f425615: Preparing
9e6784b558c3: Waiting
15dd6c63f3a2: Waiting
8819b61d0672: Waiting
30349

### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

In [ ]:
# create function
str_image_uri = '836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pd-starting-feats-2:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=60,
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': '236a1531cd30b1ab0fd51366004127badc729734901cef9256d114b23fb5f0a4',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-pd-starting-feats-2',
 'FunctionName': 'genxii-pd-starting-feats-2',
 'LastModified': '2023-10-18T22:52:48.590+0000',
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1072',
                                      'content-type': 'application/json',
                                      'date': 'Wed, 18 Oct 2023 22:52:49 GMT',
                                      'x-amzn-requestid': '1e34a623-5d16-49ba-a206-e3d58efa6fa4'},
                      'HTTPStatusCode': 201,
                      'RequestId': '1e34a623-5d16-49ba-a206-e3d58efa6fa4',
                      'RetryAttempts': 0},
 'RevisionId': '2f440e14-90e9-4787-b

### Clean-up

In [ ]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)